# Code Generation RAG Experiment Pipeline

Objective: test whether algorithmic knowledge retrieval from `cp-algorithms` improves code generation performance without using HumanEval, MBPP, or APPS as retrieval corpus.


## 1.1 Setup

Install/import dependencies, set global seeds, and define the single configuration object.


In [ ]:
import importlib.util
import os
import subprocess
import sys

REQUIRED_PACKAGES = {
    "datasets": "datasets>=2.17.0",
    "transformers": "transformers>=4.38.0",
    "torch": "torch>=2.1.0",
    "sentence_transformers": "sentence-transformers>=2.3.1",
    "rank_bm25": "rank-bm25>=0.2.2",
    "faiss": "faiss-cpu>=1.8.0",
    "numpy": "numpy>=1.24.0",
    "pandas": "pandas>=2.1.0",
    "matplotlib": "matplotlib>=3.8.0",
    "sklearn": "scikit-learn>=1.3.0",
    "scipy": "scipy>=1.11.0",
    "tqdm": "tqdm>=4.66.0",
}

AUTO_INSTALL = os.environ.get("NOTEBOOK_AUTO_INSTALL", "1") != "0"
for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        if not AUTO_INSTALL:
            raise ImportError(f"Missing dependency {pip_name}. Set NOTEBOOK_AUTO_INSTALL=1 or install it manually.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import ast
import dataclasses
import io
import json
import math
import multiprocessing as mp
import random
import re
import shutil
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from scipy.stats import chi2
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def set_global_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@dataclass
class ExperimentConfig:
    seed: int = 42
    model_name: str = os.environ.get("CODEGEN_MODEL", "Qwen/Qwen2.5-Coder-1.5B-Instruct")
    embedding_model_name: str = os.environ.get("EMBEDDING_MODEL", "flax-sentence-embeddings/st-codesearch-distilroberta-base")
    num_samples: int = int(os.environ.get("NUM_SAMPLES", "10"))
    max_tasks_per_dataset: int = int(os.environ.get("MAX_TASKS_PER_DATASET", "50"))
    apps_max_tasks: int = int(os.environ.get("APPS_MAX_TASKS", "30"))
    max_new_tokens: int = int(os.environ.get("MAX_NEW_TOKENS", "512"))
    max_input_tokens: int = int(os.environ.get("MAX_INPUT_TOKENS", "2048"))
    top_k: int = 5
    rrf_k: int = 60
    pass_k: int = 1
    do_sample: bool = os.environ.get("DO_SAMPLE", "1") != "0"
    temperature: float = float(os.environ.get("TEMPERATURE", "0.2"))
    t_min: int = 10
    tau1: float = 0.5
    tau2: float = 0.5
    entropy_threshold: float = 2.5
    stage2_similarity_threshold: float = 0.25
    retrieval_min_dense_score: float = 0.30
    rewind_tokens: int = 8
    speculative_peek_tokens: int = 20
    TPOT_ms: float = 20.0
    t_forward_ms: float = 2.0
    t_prefill_ms: float = 0.15
    t_embed_ms: float = 1.5
    t_search_ms: float = 0.2
    experiment_methods: tuple[str, ...] = ("B0_baseline", "B1_static_rag", "B2_entropy_rag", "B3_self_repair", "V1_adaptive_rag", "A1_rewind_rag", "A2_speculative_rag", "A3_entropy_query_aug")
    eval_timeout_s: int = 6
    output_jsonl: Path = Path("experiments.jsonl")
    results_json: Path = Path("results.json")
    results_csv: Path = Path("results.csv")
    plots_dir: Path = Path("artifacts/plots")
    cp_repo_url: str = "https://github.com/cp-algorithms/cp-algorithms.git"
    cp_repo_dir: Path = Path("external/cp-algorithms")
    corpus_cache_path: Path = Path("artifacts/cp_algorithms_corpus.jsonl")
    max_chunk_tokens: int = 750
    min_relevance_terms: int = 2
    relevance_overlap_threshold: float = 0.08
    allow_offline_corpus_fallback: bool = True
    torch_compile: bool = os.environ.get("TORCH_COMPILE", "0") == "1"
    oom_retry: bool = True
    run_sensitivity_analysis: bool = False
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    tokenizer: Any = field(default=None, repr=False)
    model: Any = field(default=None, repr=False)


CFG = ExperimentConfig()
set_global_seeds(CFG.seed)
CFG.plots_dir.mkdir(parents=True, exist_ok=True)
print(dataclasses.asdict(CFG) | {"tokenizer": None, "model": None})


## 1.2 Data Loading

Load HumanEval, MBPP, and APPS into the unified schema.


In [ ]:
def _parse_json_maybe(value: Any, default: Any) -> Any:
    if value is None:
        return default
    if isinstance(value, (dict, list)):
        return value
    try:
        return json.loads(value)
    except Exception:
        return default


def normalize_humaneval(row: dict[str, Any]) -> dict[str, Any]:
    entry_point = row.get("entry_point") or re.search(r"def\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(", row["prompt"]).group(1)
    test_code = row.get("test", "") + f"\ncheck({entry_point})\n"
    return {
        "task_id": str(row["task_id"]),
        "prompt": str(row["prompt"]),
        "solution": row.get("canonical_solution"),
        "difficulty": "humaneval",
        "metadata": {
            "dataset": "humaneval",
            "entry_point": entry_point,
            "test_code": test_code,
            "evaluation_source": row["prompt"],
        },
    }


def normalize_mbpp(row: dict[str, Any]) -> dict[str, Any]:
    test_list = row.get("test_list") or []
    test_setup_code = row.get("test_setup_code") or ""
    test_code = test_setup_code + "\n" + "\n".join(test_list) + "\n"
    entry_point = row.get("entry_point")
    if not entry_point:
        source = row.get("code", "") + "\n" + test_code
        match = re.search(r"def\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(", source)
        entry_point = match.group(1) if match else None
    prompt = (
        "Write Python code that satisfies the following specification. "
        "Return executable Python only.\n\n"
        f"Specification:\n{row.get('text', '')}\n"
    )
    if entry_point:
        prompt += f"\nEntry point: {entry_point}\n"
    return {
        "task_id": f"MBPP/{row.get('task_id')}",
        "prompt": prompt,
        "solution": row.get("code"),
        "difficulty": "mbpp",
        "metadata": {
            "dataset": "mbpp",
            "entry_point": entry_point,
            "test_code": test_code,
            "text": row.get("text", ""),
        },
    }


def normalize_apps(row: dict[str, Any]) -> dict[str, Any]:
    solutions = _parse_json_maybe(row.get("solutions"), [])
    solution = solutions[0] if isinstance(solutions, list) and solutions else None
    input_output = _parse_json_maybe(row.get("input_output"), {})
    prompt = str(row.get("question", ""))
    if row.get("starter_code"):
        prompt += "\n\nStarter code:\n" + str(row.get("starter_code"))
    return {
        "task_id": f"APPS/{row.get('problem_id', row.get('id', 'unknown'))}",
        "prompt": prompt,
        "solution": solution,
        "difficulty": row.get("difficulty") or "apps",
        "metadata": {
            "dataset": "apps",
            "input_output": input_output,
            "starter_code": row.get("starter_code"),
        },
    }


def load_humaneval_tasks(config: ExperimentConfig) -> list[dict[str, Any]]:
    dataset = load_dataset("openai_humaneval", split="test")
    return [normalize_humaneval(row) for row in dataset][: config.max_tasks_per_dataset]


def load_mbpp_tasks(config: ExperimentConfig) -> list[dict[str, Any]]:
    last_error = None
    for args in [("mbpp", "sanitized"), ("mbpp", None), ("google-research-datasets/mbpp", "sanitized")]:
        try:
            name, subset = args
            dataset = load_dataset(name, subset, split="test") if subset else load_dataset(name, split="test")
            return [normalize_mbpp(row) for row in dataset][: config.max_tasks_per_dataset]
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Unable to load MBPP: {last_error}")


def load_apps_tasks(config: ExperimentConfig) -> list[dict[str, Any]]:
    last_error = None
    for split in [f"test[:{config.apps_max_tasks}]", f"train[:{config.apps_max_tasks}]"]:
        try:
            dataset = load_dataset("codeparrot/apps", split=split)
            return [normalize_apps(row) for row in dataset][: config.apps_max_tasks]
        except Exception as exc:
            last_error = exc
    print(f"APPS load failed; continuing without APPS subset: {last_error}")
    return []


def load_all_tasks(config: ExperimentConfig) -> list[dict[str, Any]]:
    tasks = []
    tasks.extend(load_humaneval_tasks(config))
    tasks.extend(load_mbpp_tasks(config))
    tasks.extend(load_apps_tasks(config))
    required = {"task_id", "prompt", "solution", "difficulty"}
    for task in tasks:
        missing = required - set(task)
        if missing:
            raise ValueError(f"Task {task.get('task_id')} missing fields: {missing}")
    return tasks


TASKS = load_all_tasks(CFG)
print(pd.Series([task["metadata"]["dataset"] for task in TASKS]).value_counts())
print(f"Loaded {len(TASKS)} tasks")
TASKS[:2]


## 2. Retrieval Corpus Construction

Clone and process `cp-algorithms`; HumanEval, MBPP, and APPS are never used as retrieval corpus.


In [ ]:
CODE_BLOCK_RE = re.compile(r"```(?:cpp|c\+\+|python|py|java|text)?\n(.*?)```", re.DOTALL | re.IGNORECASE)
HEADER_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$")
COMPLEXITY_RE = re.compile(r"(?:complexit(?:y|ies).*?)(O\s*\([^\n\)]+\))", re.IGNORECASE)
TOKEN_RE = re.compile(r"[A-Za-z_][A-Za-z_0-9]+|\d+")


def clone_cp_algorithms(config: ExperimentConfig) -> None:
    if (config.cp_repo_dir / ".git").exists():
        return
    config.cp_repo_dir.parent.mkdir(parents=True, exist_ok=True)
    if config.cp_repo_dir.exists() and any(config.cp_repo_dir.iterdir()):
        raise RuntimeError(f"{config.cp_repo_dir} exists but is not a git checkout")
    subprocess.run(
        ["git", "clone", "--depth", "1", config.cp_repo_url, str(config.cp_repo_dir)],
        check=True,
    )


def estimate_tokens(text: str) -> int:
    return max(1, len(text.split()))


def get_markdown_title(text: str, path: Path) -> str:
    for line in text.splitlines():
        match = HEADER_RE.match(line)
        if match and match.group(1) == "#":
            return match.group(2).strip()
    return path.stem.replace("-", " ").replace("_", " ").title()


def split_markdown_sections(text: str, title: str) -> list[tuple[str, str]]:
    sections: list[tuple[str, list[str]]] = []
    current_section = title
    current_lines: list[str] = []
    for line in text.splitlines():
        match = HEADER_RE.match(line)
        if match and match.group(1) in {"##", "###"}:
            if current_lines:
                sections.append((current_section, current_lines))
            current_section = match.group(2).strip()
            current_lines = [line]
        else:
            current_lines.append(line)
    if current_lines:
        sections.append((current_section, current_lines))
    return [(section, "\n".join(lines).strip()) for section, lines in sections if "\n".join(lines).strip()]


def split_long_section(content: str, max_tokens: int) -> list[str]:
    if estimate_tokens(content) <= max_tokens:
        return [content]
    paragraphs = re.split(r"\n\s*\n", content)
    chunks: list[str] = []
    active: list[str] = []
    active_tokens = 0
    for paragraph in paragraphs:
        paragraph_tokens = estimate_tokens(paragraph)
        if active and active_tokens + paragraph_tokens > max_tokens:
            chunks.append("\n\n".join(active).strip())
            active = []
            active_tokens = 0
        if paragraph_tokens > max_tokens:
            words = paragraph.split()
            for start in range(0, len(words), max_tokens):
                chunks.append(" ".join(words[start : start + max_tokens]).strip())
            continue
        active.append(paragraph)
        active_tokens += paragraph_tokens
    if active:
        chunks.append("\n\n".join(active).strip())
    return [chunk for chunk in chunks if chunk]


def extract_complexity(content: str) -> Optional[str]:
    match = COMPLEXITY_RE.search(content)
    return match.group(1).replace(" ", "") if match else None


def extract_tags(path: Path, title: str, section: str) -> list[str]:
    path_tags = [part for part in path.with_suffix("").parts if part not in {".", "src", "external", "cp-algorithms"}]
    text_tags = [token.lower() for token in TOKEN_RE.findall(f"{title} {section}") if len(token) > 3]
    return sorted(set(path_tags[-4:] + text_tags[:8]))


def parse_cp_markdown_file(path: Path, repo_root: Path, config: ExperimentConfig) -> list[dict[str, Any]]:
    text = path.read_text(encoding="utf-8", errors="ignore")
    title = get_markdown_title(text, path)
    relative_path = path.relative_to(repo_root)
    entries: list[dict[str, Any]] = []
    for section, content in split_markdown_sections(text, title):
        for chunk_index, chunk in enumerate(split_long_section(content, config.max_chunk_tokens)):
            code_blocks = CODE_BLOCK_RE.findall(chunk)
            entries.append(
                {
                    "doc_id": f"{relative_path}::{section}::{chunk_index}",
                    "title": title,
                    "section": section,
                    "content": chunk,
                    "code": "\n\n".join(code_blocks).strip(),
                    "complexity": extract_complexity(chunk),
                    "tags": extract_tags(relative_path, title, section),
                    "source_path": str(relative_path),
                }
            )
    return entries


def save_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as output_file:
        for row in rows:
            output_file.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as input_file:
        return [json.loads(line) for line in input_file if line.strip()]


def offline_cp_fallback() -> list[dict[str, Any]]:
    return [
        {
            "doc_id": "offline::binary-search::0",
            "title": "Binary Search",
            "section": "Search on Answer",
            "content": "Binary search maintains an invariant over a monotonic predicate and halves the search interval each iteration.",
            "code": "int l = 0, r = n; while (l < r) { int m = (l + r) / 2; if (ok(m)) r = m; else l = m + 1; }",
            "complexity": "O(logn)",
            "tags": ["binary", "search", "monotonic"],
            "source_path": "offline-fallback",
        },
        {
            "doc_id": "offline::bfs::0",
            "title": "Breadth First Search",
            "section": "Implementation",
            "content": "Breadth first search explores graph vertices by distance layers using a queue and a visited array.",
            "code": "queue<int> q; q.push(s); used[s] = true; while (!q.empty()) { int v = q.front(); q.pop(); for (int u : adj[v]) if (!used[u]) used[u] = true, q.push(u); }",
            "complexity": "O(n+m)",
            "tags": ["graph", "bfs", "queue"],
            "source_path": "offline-fallback",
        },
    ]


def build_cp_algorithms_corpus(config: ExperimentConfig) -> list[dict[str, Any]]:
    if config.corpus_cache_path.exists():
        corpus = load_jsonl(config.corpus_cache_path)
        if corpus:
            print(f"Loaded cached cp-algorithms corpus: {len(corpus)} chunks")
            return corpus
    try:
        clone_cp_algorithms(config)
        md_files = sorted(config.cp_repo_dir.rglob("*.md"))
        corpus: list[dict[str, Any]] = []
        for path in tqdm(md_files, desc="Parsing cp-algorithms markdown"):
            corpus.extend(parse_cp_markdown_file(path, config.cp_repo_dir, config))
        if not corpus:
            raise RuntimeError("No corpus entries extracted from cp-algorithms")
        save_jsonl(config.corpus_cache_path, corpus)
        return corpus
    except Exception as exc:
        if not config.allow_offline_corpus_fallback:
            raise
        print(f"cp-algorithms clone/parse failed; using offline non-benchmark fallback corpus: {exc}")
        corpus = offline_cp_fallback()
        save_jsonl(config.corpus_cache_path, corpus)
        return corpus


CORPUS = build_cp_algorithms_corpus(CFG)
assert all(row["source_path"] != "humaneval" for row in CORPUS)
print(f"Retrieval corpus chunks: {len(CORPUS)}")
CORPUS[0]


## 3. Retrieval System

Sparse BM25, dense Sentence Transformers + FAISS, and RRF hybrid retrieval.


In [ ]:
def tokenize_for_retrieval(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_RE.findall(text)]


def corpus_text(entry: dict[str, Any]) -> str:
    return "\n".join(
        str(part)
        for part in [entry.get("title"), entry.get("section"), entry.get("content"), entry.get("code"), " ".join(entry.get("tags", []))]
        if part
    )


class HybridRetriever:
    def __init__(self, corpus: list[dict[str, Any]], config: ExperimentConfig):
        self.corpus = corpus
        self.config = config
        self.texts = [corpus_text(entry) for entry in corpus]
        self.tokenized = [tokenize_for_retrieval(text) for text in self.texts]
        self.bm25 = BM25Okapi(self.tokenized if self.tokenized else [[]])
        self.embedder = SentenceTransformer(config.embedding_model_name, device=config.device if config.device == "cuda" else "cpu")
        embedding_dim = int(self.embedder.get_sentence_embedding_dimension())
        self.embeddings = self.embedder.encode(
            self.texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=32,
            show_progress_bar=True,
        ).astype("float32") if self.texts else np.zeros((0, embedding_dim), dtype="float32")
        self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
        if len(self.embeddings):
            self.index.add(self.embeddings)
        self._query_embedding_cache: dict[str, np.ndarray] = {}

    def _embed_query(self, query: str) -> np.ndarray:
        cached = self._query_embedding_cache.get(query)
        if cached is not None:
            return cached
        embedding = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        if len(self._query_embedding_cache) >= 512:
            self._query_embedding_cache.pop(next(iter(self._query_embedding_cache)))
        self._query_embedding_cache[query] = embedding
        return embedding

    def search(self, query: str, top_k: Optional[int] = None) -> list[dict[str, Any]]:
        k = top_k or self.config.top_k
        if not query.strip() or not self.corpus:
            return []
        query_tokens = tokenize_for_retrieval(query)
        bm25_scores = np.asarray(self.bm25.get_scores(query_tokens), dtype=float)
        bm25_order = np.argsort(bm25_scores)[::-1][: max(k * 10, k)]

        query_embedding = self._embed_query(query)
        dense_scores, dense_indices = self.index.search(query_embedding, min(max(k * 10, k), len(self.corpus)))
        dense_order = dense_indices[0]
        dense_score_map = {int(idx): float(score) for idx, score in zip(dense_indices[0], dense_scores[0]) if idx >= 0}

        rrf: dict[int, float] = {}
        for rank, idx in enumerate(bm25_order):
            rrf[int(idx)] = rrf.get(int(idx), 0.0) + 1.0 / (self.config.rrf_k + rank + 1)
        for rank, idx in enumerate(dense_order):
            if idx >= 0:
                rrf[int(idx)] = rrf.get(int(idx), 0.0) + 1.0 / (self.config.rrf_k + rank + 1)

        ranked = sorted(rrf.items(), key=lambda item: item[1], reverse=True)[:k]
        results = []
        for rank, (idx, fused_score) in enumerate(ranked, start=1):
            entry = dict(self.corpus[idx])
            entry.update(
                {
                    "rank": rank,
                    "bm25_score": float(bm25_scores[idx]),
                    "dense_score": float(dense_score_map.get(idx, 0.0)),
                    "rrf_score": float(fused_score),
                }
            )
            results.append(entry)
        return results


RETRIEVER = HybridRetriever(CORPUS, CFG)
RETRIEVER.search("shortest path graph bfs", top_k=3)


## 4. Generation and Adaptive Intervention Pipelines

This section includes the two required public pipelines, `generate_baseline` and `generate_with_rag`, plus ACA-equivalent variants: `B0`, `B1`, `B2`, `B3`, `V1`, and `A1`.


In [ ]:

def prepare_generation_runtime(config: ExperimentConfig) -> ExperimentConfig:
    if config.model is not None and config.tokenizer is not None:
        return config
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    tokenizer.truncation_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if config.device == "cuda":
        model = AutoModelForCausalLM.from_pretrained(config.model_name, torch_dtype=torch.float16).to(config.device)
    else:
        model = AutoModelForCausalLM.from_pretrained(config.model_name).to(config.device)
    model.eval()
    if config.device == "cuda":
        torch.cuda.empty_cache()
    if config.torch_compile and hasattr(torch, "compile") and config.device == "cuda":
        try:
            compiled_model = torch.compile(model, mode="reduce-overhead")
            if hasattr(compiled_model, "generate"):
                model = compiled_model
        except Exception:
            pass
    config.tokenizer = tokenizer
    config.model = model
    return config


def format_prompt(prompt: str, context: Optional[str], config: ExperimentConfig, partial_solution: Optional[str] = None, rewind: bool = False) -> str:
    instruction = "Return executable Python code only."
    user_parts = [instruction]
    if context:
        user_parts.append(
            "Algorithmic reference (mostly C++/pseudocode; translate concepts to Python and do not copy C++ syntax):"
            f"\n{context}"
        )
    user_parts.append(f"Task:\n{prompt}")
    if partial_solution:
        mode = "Repair and continue this partial solution" if rewind else "Continue this partial solution"
        user_parts.append(f"{mode}:\n{partial_solution}")
    user_text = "\n\n".join(user_parts)
    tokenizer = config.tokenizer
    if tokenizer is not None and hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": "You are an expert competitive-programming and Python code-generation assistant."},
                    {"role": "user", "content": user_text},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            pass
    return user_text + "\n\nPython solution:\n"


def safe_tokenize(prompt: str, tokenizer: Any, max_length: int, device: str) -> dict[str, torch.Tensor]:
    """Left-truncate tokenized prompts so the task and partial solution at the end survive."""
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    reserve = 10
    effective_max_length = max(max_length - reserve, 1)
    if len(ids) > effective_max_length:
        ids = ids[-effective_max_length:]
    encoded = tokenizer.prepare_for_model(ids, add_special_tokens=False, return_tensors="pt")
    if "attention_mask" not in encoded:
        encoded["attention_mask"] = torch.ones_like(encoded["input_ids"])
    return {key: value.to(device) for key, value in encoded.items()}


def _generate_with_oom_retry(model: Any, inputs: dict[str, torch.Tensor], generation_kwargs: dict[str, Any], config: ExperimentConfig) -> Any:
    try:
        with torch.no_grad():
            return model.generate(**inputs, **generation_kwargs)
    except RuntimeError as exc:
        if not config.oom_retry or "out of memory" not in str(exc).lower():
            raise
        if config.device == "cuda":
            torch.cuda.empty_cache()
        retry_kwargs = dict(generation_kwargs)
        retry_kwargs["max_new_tokens"] = max(int(retry_kwargs["max_new_tokens"]) // 2, 32)
        with torch.no_grad():
            return model.generate(**inputs, **retry_kwargs)


def generate_text_with_metadata(model_prompt: str, config: ExperimentConfig, seed_offset: int = 0, return_scores: bool = False) -> tuple[str, dict[str, Any]]:
    prepare_generation_runtime(config)
    set_global_seeds(config.seed + seed_offset)
    tokenizer = config.tokenizer
    model = config.model
    inputs = safe_tokenize(model_prompt, tokenizer, config.max_input_tokens, config.device)
    generation_kwargs = {
        "max_new_tokens": config.max_new_tokens,
        "do_sample": config.do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "return_dict_in_generate": True,
        "output_scores": return_scores,
    }
    if config.do_sample:
        generation_kwargs["temperature"] = max(config.temperature, 1e-6)
    start = time.perf_counter()
    generated = _generate_with_oom_retry(model, inputs, generation_kwargs, config)
    generation_time_s = time.perf_counter() - start
    sequences = generated.sequences
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = sequences[0][prompt_len:]
    output = tokenizer.decode(new_tokens, skip_special_tokens=True)
    scores = [score[0].detach().float().cpu() for score in getattr(generated, "scores", [])] if return_scores else []
    return output, {
        "generation_time_s": generation_time_s,
        "prompt_tokens_approx": int(prompt_len),
        "output_tokens_approx": int(new_tokens.numel()),
        "generated_token_ids": new_tokens.detach().cpu().tolist(),
        "scores": scores,
    }


def generate_text(model_prompt: str, config: ExperimentConfig, seed_offset: int = 0) -> str:
    output, _ = generate_text_with_metadata(model_prompt, config, seed_offset=seed_offset, return_scores=False)
    return output


def format_retrieval_context(docs: list[dict[str, Any]], min_score: float = 0.30) -> str:
    filtered_docs = [doc for doc in docs if float(doc.get("dense_score", 0.0)) >= min_score]
    if not filtered_docs:
        return ""
    blocks = []
    for doc in filtered_docs:
        code = f"\nCode:\n{doc['code']}" if doc.get("code") else ""
        complexity = f"\nComplexity: {doc['complexity']}" if doc.get("complexity") else ""
        blocks.append(
            f"[{doc['rank']}] {doc['title']} / {doc['section']}"
            f"\nTags: {', '.join(doc.get('tags', []))}"
            f"{complexity}\nContent:\n{doc['content'][:2500]}{code}"
        )
    return "\n\n---\n\n".join(blocks)


def generate_baseline(prompt: str, config: ExperimentConfig) -> str:
    return generate_text(format_prompt(prompt, context=None, config=config), config)


def generate_with_rag(prompt: str, retriever: HybridRetriever, config: ExperimentConfig) -> str:
    docs = retriever.search(prompt, top_k=config.top_k)
    context = format_retrieval_context(docs, min_score=config.retrieval_min_dense_score)
    return generate_text(format_prompt(prompt, context=context, config=config), config)


def entropy_from_logits(logits: torch.Tensor) -> float:
    probs = torch.softmax(logits, dim=-1)
    log_probs = torch.log_softmax(logits, dim=-1)
    return float(-(probs * log_probs).sum().item())


def margin_from_logits(logits: torch.Tensor) -> float:
    probs = torch.softmax(logits, dim=-1)
    if probs.numel() < 2:
        return 1.0
    top2 = torch.topk(probs, k=2).values
    return float((top2[0] - top2[1]).item())


def cadence_positions_from_text(output_text: str, tokenizer: Any, token_ids: list[int]) -> list[int]:
    del output_text
    positions = []
    prefix = ""
    for index, token_id in enumerate(token_ids):
        prefix += tokenizer.decode([token_id], skip_special_tokens=True)
        stripped = prefix.rstrip()
        if stripped.endswith(("\n", ";", "}", ":", "return", "pass", "break", "continue")):
            positions.append(index)
    return positions or ([len(token_ids) - 1] if token_ids else [])


def find_candidate_point(token_logits_history: list[torch.Tensor], cadence_positions: list[int], t_min: int = 10) -> int:
    if not token_logits_history:
        return 0
    valid_positions = [pos for pos in cadence_positions if t_min <= pos < len(token_logits_history)]
    if not valid_positions:
        valid_positions = [pos for pos in cadence_positions if 0 <= pos < len(token_logits_history)] or [len(token_logits_history) - 1]
    entropy_history = [entropy_from_logits(logits) for logits in token_logits_history]
    return int(max(valid_positions, key=lambda pos: entropy_history[pos]))


def stage1_features(trace_meta: dict[str, Any], config: ExperimentConfig) -> dict[str, float]:
    scores = trace_meta.get("scores") or []
    token_ids = trace_meta.get("generated_token_ids") or []
    if not scores:
        return {"t_star": 0, "entropy_t": 0.0, "delta_entropy_t": 0.0, "margin_t": 1.0, "relative_pos_t": 0.0, "generated_length_t": len(token_ids)}
    cadence = cadence_positions_from_text("", config.tokenizer, token_ids)
    t_star = find_candidate_point(scores, cadence, t_min=config.t_min)
    entropy_history = [entropy_from_logits(logits) for logits in scores]
    previous_entropy = entropy_history[t_star - 1] if t_star > 0 else entropy_history[t_star]
    return {
        "t_star": int(t_star),
        "entropy_t": float(entropy_history[t_star]),
        "delta_entropy_t": float(entropy_history[t_star] - previous_entropy),
        "margin_t": float(margin_from_logits(scores[t_star])),
        "relative_pos_t": float(t_star / max(config.max_new_tokens, 1)),
        "generated_length_t": float(len(token_ids)),
    }


def sigmoid(value: float) -> float:
    return float(1.0 / (1.0 + math.exp(-max(min(value, 60.0), -60.0))))


def stage1_probability(features: dict[str, float], config: ExperimentConfig) -> float:
    centered_entropy = features["entropy_t"] - config.entropy_threshold
    logit = 1.25 * centered_entropy + 0.75 * features["delta_entropy_t"] - 2.0 * features["margin_t"] + 0.5 * features["relative_pos_t"]
    return sigmoid(logit)


def stage2_probability(top_doc: Optional[dict[str, Any]], config: ExperimentConfig) -> float:
    if not top_doc:
        return 0.0
    dense_score = float(top_doc.get("dense_score", 0.0))
    logit = 8.0 * (dense_score - config.stage2_similarity_threshold)
    return sigmoid(logit)


def compute_cost_ms(
    extra_decode_tokens: int,
    extra_forward_steps: int,
    delta_prompt_tokens: int,
    retrieval_needed: bool,
    config: ExperimentConfig,
    first_pass_tokens: int = 0,
) -> float:
    return float(
        first_pass_tokens * config.TPOT_ms
        + extra_decode_tokens * config.TPOT_ms
        + extra_forward_steps * config.t_forward_ms
        + delta_prompt_tokens * config.t_prefill_ms
        + float(bool(retrieval_needed)) * (config.t_embed_ms + config.t_search_ms)
    )


def retrieval_metadata(prompt: str, retriever: HybridRetriever, config: ExperimentConfig) -> tuple[list[dict[str, Any]], str, dict[str, Any]]:
    start = time.perf_counter()
    docs = retriever.search(prompt, top_k=config.top_k)
    retrieval_time_s = time.perf_counter() - start
    context = format_retrieval_context(docs, min_score=config.retrieval_min_dense_score)
    return docs, context, {
        "retrieval_time_s": retrieval_time_s,
        "retrieved_docs": [
            {key: doc[key] for key in ["doc_id", "rank", "title", "section", "bm25_score", "dense_score", "rrf_score"] if key in doc}
            for doc in docs
        ],
        "context_tokens_approx": len(context.split()),
    }


def make_augmented_query(prompt: str, partial_token_ids: list[int], tokenizer: Any) -> str:
    partial_text = tokenizer.decode(partial_token_ids, skip_special_tokens=True)
    last_lines = "\n".join(partial_text.splitlines()[-5:]).strip()
    return f"{prompt}\n\n# Partial code context (model got stuck here):\n{last_lines}" if last_lines else prompt


def baseline_trace(prompt: str, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    return generate_text_with_metadata(format_prompt(prompt, context=None, config=config), config, seed_offset=seed_offset, return_scores=True)


def resume_from_partial(prompt: str, partial_solution: str, context: Optional[str], config: ExperimentConfig, seed_offset: int, rewind: bool = False) -> tuple[str, dict[str, Any]]:
    continuation_prompt = format_prompt(prompt, context=context, config=config, partial_solution=partial_solution, rewind=rewind)
    continuation, meta = generate_text_with_metadata(continuation_prompt, config, seed_offset=seed_offset, return_scores=False)
    return (partial_solution + "\n" + continuation).strip(), meta


def run_b0_baseline(prompt: str, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    output, meta = baseline_trace(prompt, config, seed_offset)
    cost_ms = compute_cost_ms(0, 0, 0, False, config, first_pass_tokens=meta.get("output_tokens_approx", 0))
    return output, {**meta, "method_family": "B0", "retrieval_time_s": 0.0, "retrieved_docs": [], "context_tokens_approx": 0, "intervened": False, "cost_ms": cost_ms}


def run_b1_static_rag(prompt: str, retriever: HybridRetriever, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    docs, context, rmeta = retrieval_metadata(prompt, retriever, config)
    output, gmeta = generate_text_with_metadata(format_prompt(prompt, context=context, config=config), config, seed_offset=seed_offset, return_scores=False)
    cost_ms = compute_cost_ms(0, 0, rmeta["context_tokens_approx"], bool(docs), config, first_pass_tokens=gmeta.get("output_tokens_approx", 0))
    return output, {**gmeta, **rmeta, "method_family": "B1", "intervened": bool(context), "cost_ms": cost_ms}


def run_b2_entropy_rag(prompt: str, retriever: HybridRetriever, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    base_output, base_meta = baseline_trace(prompt, config, seed_offset)
    features = stage1_features(base_meta, config)
    first_pass_tokens = base_meta.get("output_tokens_approx", 0)
    if features["entropy_t"] < config.entropy_threshold:
        cost_ms = compute_cost_ms(0, 0, 0, False, config, first_pass_tokens=first_pass_tokens)
        return base_output, {**base_meta, **features, "method_family": "B2", "retrieval_time_s": 0.0, "retrieved_docs": [], "context_tokens_approx": 0, "intervened": False, "cost_ms": cost_ms}
    docs, context, rmeta = retrieval_metadata(prompt, retriever, config)
    prefix_ids = base_meta.get("generated_token_ids", [])[: int(features["t_star"]) + 1]
    prefix = config.tokenizer.decode(prefix_ids, skip_special_tokens=True)
    output, gmeta = resume_from_partial(prompt, prefix, context, config, seed_offset, rewind=False)
    cost_ms = compute_cost_ms(gmeta.get("output_tokens_approx", 0), 1, rmeta["context_tokens_approx"], bool(docs), config, first_pass_tokens=first_pass_tokens)
    return output, {**gmeta, **rmeta, **features, "method_family": "B2", "intervened": bool(context), "cost_ms": cost_ms, "intervention_mode": "entropy_forward_resume_proxy"}


def run_b3_self_repair(prompt: str, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    base_output, base_meta = baseline_trace(prompt, config, seed_offset)
    features = stage1_features(base_meta, config)
    prefix_ids = base_meta.get("generated_token_ids", [])[: int(features["t_star"]) + 1]
    prefix = config.tokenizer.decode(prefix_ids, skip_special_tokens=True)
    output, gmeta = resume_from_partial(prompt, prefix, context=None, config=config, seed_offset=seed_offset, rewind=True)
    cost_ms = compute_cost_ms(gmeta.get("output_tokens_approx", 0), 1, 0, False, config, first_pass_tokens=base_meta.get("output_tokens_approx", 0))
    return output, {**gmeta, **features, "method_family": "B3", "retrieval_time_s": 0.0, "retrieved_docs": [], "context_tokens_approx": 0, "intervened": True, "cost_ms": cost_ms, "intervention_mode": "self_repair_no_retrieval"}


def run_v1_adaptive_rag(prompt: str, retriever: HybridRetriever, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    base_output, base_meta = baseline_trace(prompt, config, seed_offset)
    features = stage1_features(base_meta, config)
    first_pass_tokens = base_meta.get("output_tokens_approx", 0)
    p1 = stage1_probability(features, config)
    if p1 < config.tau1:
        cost_ms = compute_cost_ms(0, 0, 0, False, config, first_pass_tokens=first_pass_tokens)
        return base_output, {**base_meta, **features, "stage1_probability": p1, "method_family": "V1", "retrieval_time_s": 0.0, "retrieved_docs": [], "context_tokens_approx": 0, "intervened": False, "cost_ms": cost_ms}
    docs, context, rmeta = retrieval_metadata(prompt, retriever, config)
    p2 = stage2_probability(docs[0] if docs else None, config)
    if p2 < config.tau2 or not context:
        cost_ms = compute_cost_ms(0, 0, 0, True, config, first_pass_tokens=first_pass_tokens)
        return base_output, {**base_meta, **rmeta, **features, "stage1_probability": p1, "stage2_probability": p2, "method_family": "V1", "intervened": False, "cost_ms": cost_ms}
    prefix_ids = base_meta.get("generated_token_ids", [])[: int(features["t_star"]) + 1]
    prefix = config.tokenizer.decode(prefix_ids, skip_special_tokens=True)
    output, gmeta = resume_from_partial(prompt, prefix, context, config, seed_offset, rewind=False)
    cost_ms = compute_cost_ms(gmeta.get("output_tokens_approx", 0), 1, rmeta["context_tokens_approx"], True, config, first_pass_tokens=first_pass_tokens)
    return output, {**gmeta, **rmeta, **features, "stage1_probability": p1, "stage2_probability": p2, "method_family": "V1", "intervened": True, "cost_ms": cost_ms, "intervention_mode": "two_stage_forward_resume_proxy"}


def run_a1_rewind_rag(prompt: str, retriever: HybridRetriever, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    base_output, base_meta = baseline_trace(prompt, config, seed_offset)
    features = stage1_features(base_meta, config)
    first_pass_tokens = base_meta.get("output_tokens_approx", 0)
    p1 = stage1_probability(features, config)
    if p1 < config.tau1:
        cost_ms = compute_cost_ms(0, 0, 0, False, config, first_pass_tokens=first_pass_tokens)
        return base_output, {**base_meta, **features, "stage1_probability": p1, "method_family": "A1", "retrieval_time_s": 0.0, "retrieved_docs": [], "context_tokens_approx": 0, "intervened": False, "cost_ms": cost_ms}
    docs, context, rmeta = retrieval_metadata(prompt, retriever, config)
    p2 = stage2_probability(docs[0] if docs else None, config)
    if p2 < config.tau2 or not context:
        cost_ms = compute_cost_ms(0, 0, 0, True, config, first_pass_tokens=first_pass_tokens)
        return base_output, {**base_meta, **rmeta, **features, "stage1_probability": p1, "stage2_probability": p2, "method_family": "A1", "intervened": False, "cost_ms": cost_ms}
    stop = max(int(features["t_star"]) + 1 - config.rewind_tokens, 0)
    prefix_ids = base_meta.get("generated_token_ids", [])[:stop]
    prefix = config.tokenizer.decode(prefix_ids, skip_special_tokens=True)
    output, gmeta = resume_from_partial(prompt, prefix, context, config, seed_offset, rewind=True)
    cost_ms = compute_cost_ms(gmeta.get("output_tokens_approx", 0) + config.rewind_tokens, 1, rmeta["context_tokens_approx"], True, config, first_pass_tokens=first_pass_tokens)
    return output, {**gmeta, **rmeta, **features, "stage1_probability": p1, "stage2_probability": p2, "method_family": "A1", "intervened": True, "cost_ms": cost_ms, "rewind_tokens": config.rewind_tokens, "intervention_mode": "rewind_ablation_proxy"}


def run_a2_speculative_rag(prompt: str, retriever: HybridRetriever, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    peek_config = dataclasses.replace(config, max_new_tokens=config.speculative_peek_tokens, do_sample=False)
    peek_output, peek_meta = generate_text_with_metadata(format_prompt(prompt, context=None, config=config), peek_config, seed_offset=seed_offset, return_scores=False)
    speculative_query = f"{prompt}\n\n# Algorithmic hint from model:\n{peek_output}"
    docs, context, rmeta = retrieval_metadata(speculative_query, retriever, config)
    output, gmeta = generate_text_with_metadata(format_prompt(prompt, context=context, config=config), config, seed_offset=seed_offset, return_scores=False)
    cost_ms = compute_cost_ms(gmeta.get("output_tokens_approx", 0), 1, rmeta["context_tokens_approx"], bool(docs), config, first_pass_tokens=peek_meta.get("output_tokens_approx", 0))
    return output, {**gmeta, **rmeta, "method_family": "A2", "speculative_peek": peek_output, "intervened": bool(context), "cost_ms": cost_ms, "intervention_mode": "speculative_algorithm_identification"}


def run_a3_entropy_query_aug(prompt: str, retriever: HybridRetriever, config: ExperimentConfig, seed_offset: int) -> tuple[str, dict[str, Any]]:
    base_output, base_meta = baseline_trace(prompt, config, seed_offset)
    features = stage1_features(base_meta, config)
    first_pass_tokens = base_meta.get("output_tokens_approx", 0)
    if features["entropy_t"] < config.entropy_threshold:
        cost_ms = compute_cost_ms(0, 0, 0, False, config, first_pass_tokens=first_pass_tokens)
        return base_output, {**base_meta, **features, "method_family": "A3", "retrieval_time_s": 0.0, "retrieved_docs": [], "context_tokens_approx": 0, "intervened": False, "cost_ms": cost_ms}
    prefix_ids = base_meta.get("generated_token_ids", [])[: int(features["t_star"]) + 1]
    augmented_query = make_augmented_query(prompt, prefix_ids, config.tokenizer)
    docs, context, rmeta = retrieval_metadata(augmented_query, retriever, config)
    prefix = config.tokenizer.decode(prefix_ids, skip_special_tokens=True)
    output, gmeta = resume_from_partial(prompt, prefix, context, config, seed_offset, rewind=False)
    cost_ms = compute_cost_ms(gmeta.get("output_tokens_approx", 0), 1, rmeta["context_tokens_approx"], bool(docs), config, first_pass_tokens=first_pass_tokens)
    return output, {**gmeta, **rmeta, **features, "method_family": "A3", "augmented_query": augmented_query, "intervened": bool(context), "cost_ms": cost_ms, "intervention_mode": "entropy_query_augmentation"}


def generate_with_rag_metadata(prompt: str, retriever: HybridRetriever, config: ExperimentConfig, seed_offset: int = 0) -> tuple[str, dict[str, Any]]:
    return run_b1_static_rag(prompt, retriever, config, seed_offset)


prepare_generation_runtime(CFG)
print(f"Loaded generator {CFG.model_name} on {CFG.device}")


## 5. Evaluation Metrics

Valid code, pass/fail, pass@k, success rate, difficulty breakdowns, time and token usage.


In [ ]:
def extract_python_code(output_text: str) -> str:
    block = re.search(r"```(?:python|py)?\n(.*?)```", output_text, re.DOTALL | re.IGNORECASE)
    if block:
        return block.group(1).strip()
    def_match = re.search(r"((?:from\s+\S+\s+import\s+.*\n|import\s+.*\n)*\s*def\s+.*)", output_text, re.DOTALL)
    if def_match:
        return def_match.group(1).strip()
    return output_text.strip()


def is_valid_python(output_text: str) -> tuple[bool, Optional[str]]:
    code = extract_python_code(output_text)
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as exc:
        return False, f"SyntaxError: {exc}"


def build_candidate_source(task: dict[str, Any], output_text: str) -> str:
    code = extract_python_code(output_text)
    dataset = task["metadata"].get("dataset")
    if dataset == "humaneval":
        if code.lstrip().startswith("def ") or code.lstrip().startswith("import ") or code.lstrip().startswith("from "):
            return code + "\n"
        body = code
        if body and not body.startswith("    "):
            body = "\n".join("    " + line if line.strip() else line for line in body.splitlines())
        return task["metadata"]["evaluation_source"] + body + "\n"
    return code + "\n"


def _exec_worker(source_code: str, test_code: str, result_queue: Any) -> None:
    namespace: dict[str, Any] = {}
    try:
        exec(source_code, namespace)
        exec(test_code, namespace)
    except Exception as exc:
        result_queue.put((False, f"{type(exc).__name__}: {exc}"))
        return
    result_queue.put((True, None))


def run_python_tests(source_code: str, test_code: str, timeout_s: int) -> tuple[bool, Optional[str]]:
    context_name = "fork" if "fork" in mp.get_all_start_methods() else "spawn"
    context = mp.get_context(context_name)
    queue = context.Queue()
    process = context.Process(target=_exec_worker, args=(source_code, test_code, queue))
    process.start()
    process.join(timeout_s)
    if process.is_alive():
        process.kill()
        process.join()
        return False, "TimeoutError"
    if queue.empty():
        return False, "NoResult"
    return queue.get()


def _apps_fn_worker(code: str, fn_name: str, inputs: list[Any], outputs: list[Any], result_queue: Any) -> None:
    namespace: dict[str, Any] = {}
    try:
        exec(code, namespace)
        fn = namespace[fn_name]
        for args, expected in zip(inputs, outputs):
            if not isinstance(args, list):
                args = [args]
            actual = fn(*args)
            if str(actual).strip() != str(expected).strip() and actual != expected:
                result_queue.put((False, f"expected={expected!r}, actual={actual!r}"))
                return
    except Exception as exc:
        result_queue.put((False, f"{type(exc).__name__}: {exc}"))
        return
    result_queue.put((True, None))


def run_apps_function_tests(code: str, payload: dict[str, Any], timeout_s: int) -> tuple[bool, Optional[str]]:
    fn_name = payload.get("fn_name")
    inputs = payload.get("inputs") or []
    outputs = payload.get("outputs") or []
    if not fn_name or not inputs or not outputs:
        return False, "No APPS function tests"
    context_name = "fork" if "fork" in mp.get_all_start_methods() else "spawn"
    context = mp.get_context(context_name)
    queue = context.Queue()
    process = context.Process(target=_apps_fn_worker, args=(code, fn_name, inputs, outputs, queue))
    process.start()
    process.join(timeout_s)
    if process.is_alive():
        process.kill()
        process.join()
        return False, "TimeoutError"
    if queue.empty():
        return False, "NoResult"
    return queue.get()


def _apps_io_text(value: Any) -> str:
    if isinstance(value, list):
        return "\n".join(str(item) for item in value)
    return str(value)


def _normalize_io_text(value: Any) -> str:
    return " ".join(_apps_io_text(value).strip().split())


def run_apps_io_tests(code: str, payload: dict[str, Any], timeout_s: int) -> tuple[bool, Optional[str]]:
    inputs = payload.get("inputs") or []
    outputs = payload.get("outputs") or []
    if not inputs or not outputs:
        return False, "No APPS stdin/stdout tests"
    for case_index, (inp, expected) in enumerate(zip(inputs, outputs)):
        input_text = _apps_io_text(inp)
        if input_text and not input_text.endswith("\n"):
            input_text += "\n"
        try:
            proc = subprocess.run(
                [sys.executable, "-c", code],
                input=input_text,
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return False, f"TimeoutError on APPS case {case_index}"
        if proc.returncode != 0:
            return False, f"RuntimeError on APPS case {case_index}: {proc.stderr.strip()}"
        actual = _normalize_io_text(proc.stdout)
        expected_text = _normalize_io_text(expected)
        if actual != expected_text:
            return False, f"APPS case {case_index}: expected={expected_text!r}, got={actual!r}"
    return True, None


def evaluate_output(task: dict[str, Any], output_text: str, config: ExperimentConfig) -> dict[str, Any]:
    valid, valid_error = is_valid_python(output_text)
    code = extract_python_code(output_text)
    dataset = task["metadata"].get("dataset")
    eval_mode = "valid_code_only"
    passed = valid
    error = valid_error
    if dataset in {"humaneval", "mbpp"} and task["metadata"].get("test_code"):
        eval_mode = "unit_tests"
        source_code = build_candidate_source(task, output_text)
        passed, error = run_python_tests(source_code, task["metadata"]["test_code"], config.eval_timeout_s)
    elif dataset == "apps":
        payload = task["metadata"].get("input_output") or {}
        if payload.get("fn_name"):
            eval_mode = "apps_fn_tests"
            passed, error = run_apps_function_tests(code, payload, config.eval_timeout_s)
        elif payload.get("inputs") and payload.get("outputs"):
            eval_mode = "apps_io_tests"
            passed, error = run_apps_io_tests(code, payload, config.eval_timeout_s)
        else:
            eval_mode = "apps_valid_code_proxy"
            passed, error = valid, valid_error
    return {
        "valid_code": bool(valid),
        "valid_code_error": valid_error,
        "passed": bool(passed),
        "error": error,
        "eval_mode": eval_mode,
        "output_tokens_approx": len(output_text.split()),
        "code_tokens_approx": len(code.split()),
    }


def estimate_pass_at_k(num_samples: int, num_correct: int, k: int) -> float:
    if num_samples <= 0:
        return 0.0
    if num_correct <= 0:
        return 0.0
    if num_samples - num_correct < k:
        return 1.0
    return float(1.0 - np.prod(1.0 - k / np.arange(num_samples - num_correct + 1, num_samples + 1)))


## 6. Experiment Loop

Checkpointed JSONL logging supports resume after interruption. The loop now runs the ACA-comparable method suite: `B0`, `B1`, `B2`, `B3`, `V1`, and `A1`.


In [ ]:

def result_key(row: dict[str, Any]) -> tuple[str, int, str]:
    return (str(row["task_id"]), int(row["sample_id"]), str(row["method"]))


def load_experiment_log(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def append_experiment_log(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as output_file:
        output_file.write(json.dumps(row, ensure_ascii=False) + "\n")


def serializable_metadata(metadata: dict[str, Any]) -> dict[str, Any]:
    clean = {}
    for key, value in metadata.items():
        if key in {"scores", "generated_token_ids"}:
            continue
        if isinstance(value, (str, int, float, bool)) or value is None:
            clean[key] = value
        elif isinstance(value, list):
            clean[key] = value
        elif isinstance(value, dict):
            clean[key] = value
        else:
            clean[key] = str(value)
    return clean


def make_log_entry(
    task: dict[str, Any],
    sample_id: int,
    method: str,
    output: str,
    generation_metadata: dict[str, Any],
    eval_metadata: dict[str, Any],
) -> dict[str, Any]:
    return {
        "task_id": task["task_id"],
        "sample_id": sample_id,
        "method": method,
        "output": output,
        "metadata": {
            "dataset": task["metadata"].get("dataset"),
            "difficulty": task["difficulty"],
            "model_name": CFG.model_name,
            "seed": CFG.seed + sample_id,
            **serializable_metadata(generation_metadata),
            **eval_metadata,
        },
    }


def run_one_method(task: dict[str, Any], sample_id: int, method: str, retriever: HybridRetriever, config: ExperimentConfig) -> dict[str, Any]:
    seed_offset = sample_id
    if method == "B0_baseline":
        output, generation_metadata = run_b0_baseline(task["prompt"], config, seed_offset)
    elif method == "B1_static_rag":
        output, generation_metadata = run_b1_static_rag(task["prompt"], retriever, config, seed_offset)
    elif method == "B2_entropy_rag":
        output, generation_metadata = run_b2_entropy_rag(task["prompt"], retriever, config, seed_offset)
    elif method == "B3_self_repair":
        output, generation_metadata = run_b3_self_repair(task["prompt"], config, seed_offset)
    elif method == "V1_adaptive_rag":
        output, generation_metadata = run_v1_adaptive_rag(task["prompt"], retriever, config, seed_offset)
    elif method == "A1_rewind_rag":
        output, generation_metadata = run_a1_rewind_rag(task["prompt"], retriever, config, seed_offset)
    elif method == "A2_speculative_rag":
        output, generation_metadata = run_a2_speculative_rag(task["prompt"], retriever, config, seed_offset)
    elif method == "A3_entropy_query_aug":
        output, generation_metadata = run_a3_entropy_query_aug(task["prompt"], retriever, config, seed_offset)
    elif method == "baseline":
        output, generation_metadata = run_b0_baseline(task["prompt"], config, seed_offset)
    elif method == "rag":
        output, generation_metadata = run_b1_static_rag(task["prompt"], retriever, config, seed_offset)
    else:
        raise ValueError(f"Unknown method: {method}")
    eval_metadata = evaluate_output(task, output, config)
    return make_log_entry(task, sample_id, method, output, generation_metadata, eval_metadata)


def run_experiment(tasks: list[dict[str, Any]], retriever: HybridRetriever, config: ExperimentConfig) -> list[dict[str, Any]]:
    existing_rows = load_experiment_log(config.output_jsonl)
    completed = {result_key(row) for row in existing_rows}
    methods = list(config.experiment_methods)
    total = len(tasks) * config.num_samples * len(methods)
    expected_keys = {(task["task_id"], sample_id, method) for task in tasks for sample_id in range(config.num_samples) for method in methods}
    with tqdm(total=total, desc="Experiment") as progress:
        progress.update(len(completed & expected_keys))
        for task in tasks:
            for sample_id in range(config.num_samples):
                for method in methods:
                    key = (task["task_id"], sample_id, method)
                    if key in completed:
                        continue
                    try:
                        row = run_one_method(task, sample_id, method, retriever, config)
                    except Exception as exc:
                        row = make_log_entry(
                            task,
                            sample_id,
                            method,
                            output="",
                            generation_metadata={"generation_time_s": 0.0, "retrieval_time_s": 0.0, "retrieved_docs": [], "failed": True, "cost_ms": 0.0},
                            eval_metadata={"valid_code": False, "passed": False, "error": f"{type(exc).__name__}: {exc}", "eval_mode": "generation_error", "output_tokens_approx": 0, "code_tokens_approx": 0},
                        )
                    append_experiment_log(config.output_jsonl, row)
                    completed.add(key)
                    progress.update(1)
    return [row for row in load_experiment_log(config.output_jsonl) if result_key(row) in expected_keys]


EXPERIMENT_ROWS = run_experiment(TASKS, RETRIEVER, CFG)
print(f"Logged rows: {len(EXPERIMENT_ROWS)} -> {CFG.output_jsonl}")


## 7. Retrieval Evaluation

Compute Recall@k, MRR@k, and downstream pass@k impact.


In [ ]:
STOP_WORDS = {
    "the", "and", "for", "with", "that", "this", "from", "return", "true", "false", "none",
    "write", "function", "given", "list", "array", "integer", "python", "code", "input", "output",
}


def relevance_terms(text: str) -> set[str]:
    return {token.lower() for token in TOKEN_RE.findall(text or "") if len(token) > 2 and token.lower() not in STOP_WORDS}


def relevant_doc_ids_for_task(task: dict[str, Any], corpus: list[dict[str, Any]], config: ExperimentConfig) -> set[str]:
    query_terms = relevance_terms((task.get("solution") or "") + "\n" + task["prompt"])
    if not query_terms:
        return set()
    relevant = set()
    for entry in corpus:
        doc_terms = relevance_terms(corpus_text(entry))
        overlap = query_terms & doc_terms
        score = len(overlap) / max(1, len(query_terms))
        if len(overlap) >= config.min_relevance_terms and score >= config.relevance_overlap_threshold:
            relevant.add(entry["doc_id"])
    return relevant


def compute_retrieval_metrics(tasks: list[dict[str, Any]], retriever: HybridRetriever, corpus: list[dict[str, Any]], config: ExperimentConfig, k: int) -> dict[str, Any]:
    recalls = []
    reciprocal_ranks = []
    evaluated = 0
    skipped = 0
    for task in tasks:
        relevant = relevant_doc_ids_for_task(task, corpus, config)
        if not relevant:
            skipped += 1
            continue
        evaluated += 1
        retrieved = retriever.search(task["prompt"], top_k=k)
        retrieved_ids = [doc["doc_id"] for doc in retrieved]
        hits = [idx for idx, doc_id in enumerate(retrieved_ids, start=1) if doc_id in relevant]
        recalls.append(len(set(retrieved_ids) & relevant) / len(relevant))
        reciprocal_ranks.append(1.0 / hits[0] if hits else 0.0)
    return {
        "recall_at_k": float(np.mean(recalls)) if recalls else 0.0,
        "mrr_at_k": float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0,
        "evaluated_tasks": evaluated,
        "skipped_tasks_without_weak_relevance": skipped,
        "k": k,
    }


def mcnemar_test(method_a_results: list[bool], method_b_results: list[bool]) -> dict[str, Any]:
    b = sum(1 for a, b_ in zip(method_a_results, method_b_results) if a and not b_)
    c = sum(1 for a, b_ in zip(method_a_results, method_b_results) if not a and b_)
    if b + c == 0:
        return {"b": b, "c": c, "statistic": 0.0, "p_value": 1.0, "significant": False}
    statistic = float((abs(b - c) - 1) ** 2 / (b + c))
    p_value = float(1.0 - chi2.cdf(statistic, df=1))
    return {"b": b, "c": c, "statistic": statistic, "p_value": p_value, "significant": p_value < 0.05}


def compute_pairwise_significance(df: pd.DataFrame, baseline: str = "B0_baseline") -> pd.DataFrame:
    if df.empty or baseline not in set(df["method"]):
        return pd.DataFrame(columns=["method", "b", "c", "statistic", "p_value", "significant", "n_shared_tasks"])
    task_success = df.groupby(["method", "task_id"])["passed"].any().reset_index()
    baseline_series = task_success[task_success["method"] == baseline].set_index("task_id")["passed"]
    rows = []
    for method in sorted(task_success["method"].unique()):
        if method == baseline:
            continue
        method_series = task_success[task_success["method"] == method].set_index("task_id")["passed"]
        shared = baseline_series.index.intersection(method_series.index)
        result = mcnemar_test(baseline_series.loc[shared].tolist(), method_series.loc[shared].tolist())
        rows.append({"method": method, "n_shared_tasks": int(len(shared)), **result})
    return pd.DataFrame(rows)


def tau_sensitivity_analysis(tasks_val: list[dict[str, Any]], retriever: HybridRetriever, config: ExperimentConfig) -> dict[str, dict[str, float]]:
    results = {}
    for tau1 in np.arange(0.2, 0.91, 0.1):
        for tau2 in np.arange(0.2, 0.91, 0.1):
            cfg = dataclasses.replace(
                config,
                tau1=float(round(tau1, 2)),
                tau2=float(round(tau2, 2)),
                experiment_methods=("V1_adaptive_rag",),
                output_jsonl=Path(f"experiments_tau_sensitivity_t1_{tau1:.1f}_t2_{tau2:.1f}.jsonl"),
            )
            rows = run_experiment(tasks_val, retriever, cfg)
            pass_rate = float(np.mean([row["metadata"].get("passed", False) for row in rows])) if rows else 0.0
            activation = float(np.mean([row["metadata"].get("intervened", False) for row in rows])) if rows else 0.0
            results[f"tau1={tau1:.1f},tau2={tau2:.1f}"] = {"pass_rate": pass_rate, "activation_rate": activation}
    return results


def summarize_results(rows: list[dict[str, Any]], config: ExperimentConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    df = pd.DataFrame([
        {
            "task_id": row["task_id"],
            "sample_id": row["sample_id"],
            "method": row["method"],
            "difficulty": row["metadata"].get("difficulty"),
            "dataset": row["metadata"].get("dataset"),
            "passed": bool(row["metadata"].get("passed", False)),
            "valid_code": bool(row["metadata"].get("valid_code", False)),
            "generation_time_s": float(row["metadata"].get("generation_time_s", 0.0)),
            "retrieval_time_s": float(row["metadata"].get("retrieval_time_s", 0.0)),
            "output_tokens_approx": int(row["metadata"].get("output_tokens_approx", 0)),
            "context_tokens_approx": int(row["metadata"].get("context_tokens_approx", 0)),
            "cost_ms": float(row["metadata"].get("cost_ms", 0.0)),
            "intervened": bool(row["metadata"].get("intervened", False)),
        }
        for row in rows
    ])
    method_rows = []
    for method, method_df in df.groupby("method"):
        pass_values = []
        for _, task_df in method_df.groupby("task_id"):
            n = len(task_df)
            c = int(task_df["passed"].sum())
            pass_values.append(estimate_pass_at_k(n, c, min(config.pass_k, n)))
        method_rows.append(
            {
                "method": method,
                "pass@k": float(np.mean(pass_values)) if pass_values else 0.0,
                "success_rate": float(method_df["passed"].mean()) if len(method_df) else 0.0,
                "valid_code_rate": float(method_df["valid_code"].mean()) if len(method_df) else 0.0,
                "mean_inference_time_s": float((method_df["generation_time_s"] + method_df["retrieval_time_s"]).mean()) if len(method_df) else 0.0,
                "mean_generation_time_s": float(method_df["generation_time_s"].mean()) if len(method_df) else 0.0,
                "mean_retrieval_time_s": float(method_df["retrieval_time_s"].mean()) if len(method_df) else 0.0,
                "mean_output_tokens_approx": float(method_df["output_tokens_approx"].mean()) if len(method_df) else 0.0,
                "mean_context_tokens_approx": float(method_df["context_tokens_approx"].mean()) if len(method_df) else 0.0,
                "mean_cost_ms": float(method_df["cost_ms"].mean()) if len(method_df) else 0.0,
                "activation_rate": float(method_df["intervened"].mean()) if len(method_df) else 0.0,
            }
        )
    summary_df = pd.DataFrame(method_rows).sort_values("method")
    difficulty_df = (
        df.groupby(["method", "difficulty"], dropna=False)
        .agg(success_rate=("passed", "mean"), valid_code_rate=("valid_code", "mean"), n=("passed", "size"))
        .reset_index()
    )
    retrieval_metrics = compute_retrieval_metrics(TASKS, RETRIEVER, CORPUS, config, config.top_k)
    significance_df = compute_pairwise_significance(df, baseline="B0_baseline")
    sensitivity_results = tau_sensitivity_analysis(TASKS[: min(5, len(TASKS))], RETRIEVER, config) if config.run_sensitivity_analysis else {}
    payload = {
        "summary": summary_df.to_dict(orient="records"),
        "by_difficulty": difficulty_df.to_dict(orient="records"),
        "retrieval": retrieval_metrics,
        "significance": significance_df.to_dict(orient="records"),
        "tau_sensitivity": sensitivity_results,
        "config": {
            field.name: (str(getattr(config, field.name)) if isinstance(getattr(config, field.name), Path) else getattr(config, field.name))
            for field in dataclasses.fields(config)
            if field.name not in {"tokenizer", "model"}
        },
    }
    config.results_json.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    summary_df.to_csv(config.results_csv, index=False)
    return summary_df, difficulty_df, significance_df, payload


SUMMARY_DF, DIFFICULTY_DF, SIGNIFICANCE_DF, RESULTS_PAYLOAD = summarize_results(EXPERIMENT_ROWS, CFG)
print(json.dumps(RESULTS_PAYLOAD["retrieval"], indent=2))
SUMMARY_DF


## 8. Tables, Plots, and Exports

Baseline vs RAG table, performance by difficulty, cost vs accuracy, JSON and CSV exports.


In [ ]:
display(SUMMARY_DF)
display(DIFFICULTY_DF)
display(SIGNIFICANCE_DF)

pivot = DIFFICULTY_DF.pivot(index="difficulty", columns="method", values="success_rate").fillna(0.0)
ax = pivot.plot(kind="bar", figsize=(10, 4), title="Performance vs Difficulty")
ax.set_ylabel("Success rate")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
performance_plot_path = CFG.plots_dir / "performance_vs_difficulty.png"
plt.savefig(performance_plot_path, dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
for _, row in SUMMARY_DF.iterrows():
    cost = row["mean_cost_ms"] if "mean_cost_ms" in row else row["mean_inference_time_s"]
    accuracy = row["success_rate"]
    ax.scatter(cost, accuracy, s=120, label=row["method"])
    ax.annotate(row["method"], (cost, accuracy), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("Mean intervention cost proxy (ms)")
ax.set_ylabel("Success rate")
ax.set_title("Cost vs Accuracy")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
cost_plot_path = CFG.plots_dir / "cost_vs_accuracy.png"
plt.savefig(cost_plot_path, dpi=160)
plt.show()

print(f"JSON results: {CFG.results_json.resolve()}")
print(f"CSV summary: {CFG.results_csv.resolve()}")
print(f"Plots: {performance_plot_path.resolve()}, {cost_plot_path.resolve()}")
